In [5]:
import sqlite3
import re

def init_db(db_path="aldi_products.db"):
    """Creates the database and tables if they don't exist."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS products (
            sku TEXT PRIMARY KEY,
            name TEXT,
            brand_name TEXT,
            selling_size TEXT,
            price_pennies INTEGER,
            comparison_price_display TEXT,
            description TEXT,
            image_url TEXT,
            last_updated DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS product_categories (
            sku TEXT,
            category_id TEXT,
            category_name TEXT,
            PRIMARY KEY (sku, category_id),
            FOREIGN KEY(sku) REFERENCES products(sku)
        )
    ''')
    conn.commit()
    return conn

def insert_product(conn, api_data):
    """Parses the raw JSON API data and inserts it into the database safely."""
    cursor = conn.cursor()
    
    # 1. Safely extract core values
    sku = api_data.get("sku")
    name = api_data.get("name")
    brand = api_data.get("brandName")
    size = api_data.get("sellingSize")
    desc = api_data.get("description")
    
    # Extract price and convert to pennies (e.g., 55)
    price_dict = api_data.get("price", {})
    price_pennies = price_dict.get("amount", 0) 
    comparison = price_dict.get("comparisonDisplay")
    
    # 2. Reconstruct the image URL from assets if available
    image_url = None
    assets = api_data.get("assets", [])
    if assets:
        raw_url = assets[0].get("url", "")
        slug = api_data.get("urlSlugText", "")
        # Replace template tokens with real values (using 400 as a standard width)
        image_url = raw_url.replace("{width}", "400").replace("{slug}", slug)

    # 3. Insert or Update the main product row
    cursor.execute('''
        INSERT INTO products (sku, name, brand_name, selling_size, price_pennies, comparison_price_display, description, image_url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(sku) DO UPDATE SET
            name=excluded.name,
            brand_name=excluded.brand_name,
            selling_size=excluded.selling_size,
            price_pennies=excluded.price_pennies,
            comparison_price_display=excluded.comparison_price_display,
            description=excluded.description,
            image_url=excluded.image_url,
            last_updated=CURRENT_TIMESTAMP
    ''', (sku, name, brand, size, price_pennies, comparison, desc, image_url))

    # 4. Handle Categories
    categories = api_data.get("categories", [])
    for cat in categories:
        cursor.execute('''
            INSERT OR IGNORE INTO product_categories (sku, category_id, category_name)
            VALUES (?, ?, ?)
        ''', (sku, cat.get("id"), cat.get("name")))

    conn.commit()

# --- Example Usage ---
# Assuming 'raw_api_response' contains the exact JSON object you pasted:
conn = init_db()
conn.close()

In [3]:
!pip install curl_cffi

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ --------------------- 0.8/1.7 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 8.8 MB/s  0:00:00


In [4]:
import xml.etree.ElementTree as ET
import re
# Use curl_cffi instead of standard requests
from curl_cffi import requests

def fetch_and_parse_protected_sitemap(sitemap_url):
    print(f"Requesting live sitemap via curl_cffi from: {sitemap_url}...")
    
    # 1. 'impersonate' acts exactly like a regular Chrome browser at the TLS/network level
    response = requests.get(sitemap_url, impersonate="chrome", stream=True)
    
    if response.status_code != 200:
        print(f"Failed to fetch sitemap. Status code: {response.status_code}")
        return []

    # 2. Extract the binary generator block to pass to the XML parser
    raw_stream = response.iter_content(chunk_size=65536)
    
    # Python's ET.iterparse expects an object with a read() method, 
    # so we wrap the generator using a simple raw stream helper class.
    class StreamWrapper:
        def __init__(self, generator):
            self.gen = generator
        def read(self, size=-1):
            try:
                return next(self.gen)
            except StopIteration:
                return b""

    # 3. Process the XML stream on-the-fly
    context = ET.iterparse(StreamWrapper(raw_stream), events=("start", "end"))
    
    sku_pattern = re.compile(r"(\d+)$")
    extracted_products = []
    namespace = ""
    current_loc = None
    current_lastmod = None
    
    for event, elem in context:
        if event == "start" and not namespace and elem.tag.startswith("{"):
            namespace = elem.tag.split("}")[0] + "}"
            
        if event == "end":
            if elem.tag == f"{namespace}loc":
                current_loc = elem.text.strip() if elem.text else None
            elif elem.tag == f"{namespace}lastmod":
                current_lastmod = elem.text.strip() if elem.text else None
                
            elif elem.tag == f"{namespace}url":
                if current_loc:
                    match = sku_pattern.search(current_loc)
                    if match:
                        extracted_products.append({
                            "sku": match.group(1),
                            "lastmod": current_lastmod,
                            "url": current_loc
                        })
                elem.clear()
                current_loc = None
                current_lastmod = None
                
    return extracted_products

TARGET_URL = "https://www.aldi.co.uk/sitemap_products.xml"
products = fetch_and_parse_protected_sitemap(TARGET_URL)

print(f"\nExtraction complete! Discovered {len(products)} products.")
if products:
    print(f"Sample Entry: {products[0]}")

Requesting live sitemap via curl_cffi from: https://www.aldi.co.uk/sitemap_products.xml...

Extraction complete! Discovered 4830 products.
Sample Entry: {'sku': '000000000000404734', 'lastmod': '2026-03-12', 'url': 'https://www.aldi.co.uk/product/specially-selected-finely-sliced-wiltshire-ham-000000000000404734'}


In [7]:
import sqlite3

# Connect to your existing database file
conn = sqlite3.connect("aldi_products.db")
cursor = conn.cursor()

try:
    # Manually inject the missing column into the products table structure
    cursor.execute("ALTER TABLE products ADD COLUMN lastmod_sitemap TEXT;")
    conn.commit()
    print("Successfully added the 'lastmod_sitemap' column to your database!")
except sqlite3.OperationalError:
    print("The column might already exist or the table structure is locked.")
finally:
    conn.close()

Successfully added the 'lastmod_sitemap' column to your database!


In [9]:
import sqlite3
import time
import random
from curl_cffi import requests

DB_PATH = "aldi_products.db"

def init_db():
    """Initializes the database and ensures tables are ready."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Core products table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS products (
            sku TEXT PRIMARY KEY,
            name TEXT,
            brand_name TEXT,
            selling_size TEXT,
            price_pennies INTEGER,
            comparison_price_display TEXT,
            description TEXT,
            image_url TEXT,
            lastmod_sitemap TEXT,        -- Tracked to check if we need to re-scrape
            last_updated DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Categories helper table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS product_categories (
            sku TEXT,
            category_id TEXT,
            category_name TEXT,
            PRIMARY KEY (sku, category_id),
            FOREIGN KEY(sku) REFERENCES products(sku)
        )
    ''')
    conn.commit()
    return conn

def get_already_scraped_skus(conn):
    """
    Returns a dictionary of {sku: lastmod_sitemap} for products already in the DB.
    This lets us skip items that haven't been updated by Aldi since our last run.
    """
    cursor = conn.cursor()
    cursor.execute("SELECT sku, lastmod_sitemap FROM products")
    return {row[0]: row[1] for row in cursor.fetchall()}

def fetch_product_json(sku):
    """Fetches raw product data from the Aldi API using browser impersonation."""
    api_url = f"https://api.aldi.co.uk/v2/products/{sku}"
    
    try:
        # Impersonating a modern desktop Chrome browser to bypass Akamai
        response = requests.get(api_url, impersonate="chrome", timeout=10)
        
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 404:
            print(f" SKU {sku} returned 404 (Not Found). Skipping.")
            return None
        elif response.status_code == 403:
            print("🛑 Blocked (403 Forbidden). We need to back off or check headers.")
            return "BLOCKED"
        else:
            print(f" Failed to fetch SKU {sku}. Status Code: {response.status_code}")
            return None
    except Exception as e:
        print(f" Error fetching SKU {sku}: {e}")
        return None

def save_product_to_db(conn, api_data, sitemap_lastmod):
    """Parses and securely inserts or updates the product payload in SQLite."""
    cursor = conn.cursor()
    
    # Safely unpack the 'data' layer if present, else fallback to root
    product_core = api_data.get("data", api_data) if "data" in api_data else api_data
    
    sku = product_core.get("sku")
    name = product_core.get("name")
    brand = product_core.get("brandName")
    size = product_core.get("sellingSize")
    desc = product_core.get("description")
    
    # Extract price structures safely
    price_dict = product_core.get("price")
    
    # If price is explicitly None/null or completely missing, fall back to an empty dict
    if not price_dict: 
        price_dict = {}
        
    price_pennies = price_dict.get("amount", 0)
    comparison = price_dict.get("comparisonDisplay")
    
    # Reconstruct Image URL string format if assets are active
    image_url = None
    assets = product_core.get("assets", [])
    if assets:
        raw_url = assets[0].get("url", "")
        slug = product_core.get("urlSlugText", "")
        image_url = raw_url.replace("{width}", "400").replace("{slug}", slug)

    # Upsert data into core products table
    cursor.execute('''
        INSERT INTO products (sku, name, brand_name, selling_size, price_pennies, comparison_price_display, description, image_url, lastmod_sitemap)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(sku) DO UPDATE SET
            name=excluded.name,
            brand_name=excluded.brand_name,
            selling_size=excluded.selling_size,
            price_pennies=excluded.price_pennies,
            comparison_price_display=excluded.comparison_price_display,
            description=excluded.description,
            image_url=excluded.image_url,
            lastmod_sitemap=excluded.lastmod_sitemap,
            last_updated=CURRENT_TIMESTAMP
    ''', (sku, name, brand, size, price_pennies, comparison, desc, image_url, sitemap_lastmod))

    # Insert nested categories safely
    categories = product_core.get("categories", [])
    for cat in categories:
        cursor.execute('''
            INSERT OR IGNORE INTO product_categories (sku, category_id, category_name)
            VALUES (?, ?, ?)
        ''', (sku, cat.get("id"), cat.get("name")))

    conn.commit()

def run_pipeline(discovered_products):
    """
    Main logic coordinator. Takes the list of dicts from your sitemap parser:
    [{'sku': '000000000383372001', 'lastmod': '2026-03-12', 'url': '...'}, ...]
    """
    conn = init_db()
    existing_records = get_already_scraped_skus(conn)
    
    print(f"Starting API extraction loop for {len(discovered_products)} potential items...")
    
    consecutive_blocks = 0
    
    for idx, item in enumerate(discovered_products):
        sku = item["sku"]
        sitemap_lastmod = item["lastmod"]
        
        # Smart Check: Has this item been scanned before, and has it changed since?
        if sku in existing_records and existing_records[sku] == sitemap_lastmod:
            # Skip fetching because data is already completely current
            continue
            
        print(f"[{idx+1}/{len(discovered_products)}] Fetching details for SKU: {sku}...")
        
        # Hit the target endpoint
        api_payload = fetch_product_json(sku)
        
        if api_payload == "BLOCKED":
            consecutive_blocks += 1
            if consecutive_blocks >= 3:
                print("🚨 API is consistently blocking us. Halting script execution to protect your IP address.")
                break
            time.sleep(10) # Wait a bit before trying again
            continue
            
        consecutive_blocks = 0 # reset counter on success
        
        if api_payload:
            save_product_to_db(conn, api_payload, sitemap_lastmod)
            print(f" ✓ Successfully stored product: {api_payload.get('name', sku)}")
            
        # Throttling Guardrail: Crucial to remain under the detection radar!
        # Delays individual API queries by a random window between 1.5 and 3.5 seconds.
        time.sleep(random.uniform(1.5, 3.5))
        
    conn.close()
    print("Pipeline routine finished execution.")

# --- How to wire this directly underneath your existing Sitemap Script: ---
# 1. Let your sitemap fetch and fill the 'products' array list variables.
# 2. Fire the run_pipeline execution block:
run_pipeline(products)

Starting API extraction loop for 4830 potential items...
[50/4830] Fetching details for SKU: 000000000000602164...
 ✓ Successfully stored product: 000000000000602164
[123/4830] Fetching details for SKU: 000000000000416555...
 ✓ Successfully stored product: 000000000000416555
[124/4830] Fetching details for SKU: 000000000000416860...
 ✓ Successfully stored product: 000000000000416860
[125/4830] Fetching details for SKU: 000000000000417638...
 ✓ Successfully stored product: 000000000000417638
[126/4830] Fetching details for SKU: 000000000000422259...
 ✓ Successfully stored product: 000000000000422259
[127/4830] Fetching details for SKU: 000000000000422948...
 ✓ Successfully stored product: 000000000000422948
[128/4830] Fetching details for SKU: 000000000000423201...
 ✓ Successfully stored product: 000000000000423201
[129/4830] Fetching details for SKU: 000000000000423411...
 ✓ Successfully stored product: 000000000000423411
[130/4830] Fetching details for SKU: 000000000000423412...
 ✓ Su